# Lab 4 — DevOps: modular code and unit tests (Course 4)
Lessons 4.1, 4.4 · exercises `4a`–`4c` · project folder `lab4_project/`

```
lab4_project/
├── pytest.ini              # pytest settings (pythonpath = .)
├── src/transforms.py       # ← 4a: write your functions here
└── tests/
    ├── conftest.py         # shared `spark` fixture
    └── test_transforms.py  # ← 4b/4c: write your tests here
```
Open the files in VS Code, edit them, then come back and run the cells. This is the same layout as the course's
`src/helpers/project_functions.py` + `tests/unit_tests/`.

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
import subprocess, importlib
PROJECT = os.path.abspath("../lab4_project")
def pytest(*args):
    r = subprocess.run([sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", *args],
                       cwd=PROJECT, capture_output=True, text=True, env={**os.environ, "PYTHONPATH": os.path.abspath("..")})
    print(r.stdout[-4000:], r.stderr[-2000:])

## 4a · Modularize PySpark code (lesson 4.1)
Course "before" code (everything in one block):
```python
df = spark.read.table("samples.nyctaxi.trips").withColumn("trip_distance_km", F.round(F.col("trip_distance") * 1.60934, 2))
df = df.select([F.col(c).alias(c.upper()) for c in df.columns])
```
**Task:** implement the 3 functions in `lab4_project/src/transforms.py` (docstrings explain each one).
Small functions = easy to **reuse** and easy to **unit test**.

In [ ]:
sys.path.insert(0, PROJECT)
import src.transforms as transforms
importlib.reload(transforms)
trips = spark.createDataFrame([(1.0, 0), (2.5, 1), (10.0, 2), (3.3, None)], ["trip_distance", "HighCholest"])
try:
    show(transforms.add_high_cholest_flag(transforms.uppercase_column_names(
         transforms.convert_miles_to_km(trips, "trip_distance_km", "trip_distance")).withColumnRenamed("HIGHCHOLEST", "HighCholest")))
except NotImplementedError:
    print("Not implemented yet — edit lab4_project/src/transforms.py")
check("4a")

## 4b · Unit tests with pytest (lesson 4.4)
- pytest finds files `test_*.py` and functions `test_*` automatically.
- `pyspark.testing.utils.assertDataFrameEqual(actual, expected)` compares DataFrames (rows + schema);
  `assertSchemaEqual` compares only schemas.
- The `spark` fixture in `conftest.py` is created once and shared by all tests.

**Task:** add tests to `lab4_project/tests/test_transforms.py` (at least 3 tests in total).

In [ ]:
pytest()
check("4b")

## 4c · Do your tests really catch bugs?
The checker runs your tests against **6 buggy versions** of `transforms.py` (in `labutils/_mutants/`).
Good tests **fail** on every buggy version. Add tests for edge cases until no bug "survives".

In [ ]:
check("4c")

## Concepts to know (no local lab)
**Integration tests (4.5)** — two ways: SDP expectations on test tables (`ON VIOLATION FAIL UPDATE`), or a Lakeflow
Job with tasks `Unit_Tests → ETL pipeline → integration test notebook`.

**Git folders (4.6)** — clone a GitHub repo into the workspace, branch, commit, push; merge with a pull request in GitHub;
then *pull* in Databricks. Authenticate with a Personal Access Token.

**Declarative Automation Bundles (4.7)** — read `lab4_project/databricks.yml`: one YAML file describes the job,
and **targets** `dev` / `prod` change only the settings. Commands: `databricks bundle validate`,
`databricks bundle deploy -t dev`, `databricks bundle run -t dev health_etl_job`.

In [ ]:
print(open(os.path.join(PROJECT, "databricks.yml")).read())